In [1]:
# Setting system path and project root
import os
import sys

PROJECT_ROOT_DIR = os.path.abspath('../../../')
sys.path.append(PROJECT_ROOT_DIR) # bringing system path to project root

In [2]:
import pandas as pd

csv_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/fineweb2_metadata/fineweb2-language-distribution.csv")
fw2_lang_dist_df = pd.read_csv(csv_path)

# Remove rows where the "subset" column ends with "_removed"
mask = (
    ~fw2_lang_dist_df["subset"].astype(str).str.endswith("_removed") &
    (fw2_lang_dist_df["split"] == "train")
)
fw2_lang_dist_df = fw2_lang_dist_df.loc[mask].reset_index(drop=True)

columns_of_interest = ["code", "name", "script", "family"]

for col in columns_of_interest:
    if col in fw2_lang_dist_df.columns:
        uniques = fw2_lang_dist_df[col].dropna().unique()   # drop NaN, then get uniques
        print(f"\nUnique values in '{col}' ({len(uniques)} total):")
        print(uniques)
    else:
        print(f"\nColumn '{col}' not found in the CSV.")


Unique values in 'code' (1812 total):
['aai' 'aak' 'aau' ... 'zyb' 'zyp' 'und']

Unique values in 'name' (1811 total):
['Arifama-Miniafia' 'Ankave' 'Abau' ... 'Yongbei Zhuang' 'Zyphe Chin'
 'Undetermined language']

Unique values in 'script' (59 total):
['Latn' 'Cyrl' 'Arab' 'Syrc' 'Ethi' 'Deva' 'Beng' 'Mymr' 'Tibt' 'Thai'
 'Cher' 'Hani' 'Copt' 'Cans' 'Thaa' 'Grek' 'Goth' 'Gujr' 'Hebr' 'Armn'
 'Jpan' 'Orya' 'Knda' 'Geor' 'Telu' 'Khmr' 'Hang' 'Kali' 'Laoo' 'Limb'
 'Lisu' 'Mlym' 'Mtei' 'Nkoo' 'Guru' 'Olck' 'Sinh' 'Mong' 'Taml' 'Tfng'
 'Kana' 'Sgnw' 'Hira' 'Hluw' 'Bamu' 'Newa' 'Xsux' 'Hung' 'Lina' 'Brai'
 'Shaw' 'Yiii' 'Vaii' 'Bopo' 'Tang' 'Sylo' 'Bali' 'Lana' 'Egyp']

Unique values in 'family' (103 total):
['Austronesian' 'Trans-New Guinea' 'Sepik' 'Niger-Congo' 'Abkhaz-Adyghe'
 'Creole' 'Maipurean' 'Nilo-Saharan' 'Afro-Asiatic' 'Sino-Tibetan' 'Mayan'
 'Jivaroan' 'Australian' 'Indo-European' 'Senagi' 'Nakh-Daghestanian'
 'Language isolate' 'Cariban' 'Algic' 'Turkic' 'Arai (Left May)' 'B

In [3]:
# Joel: took 190 languages, referred to the languages that can be evaluated (through lm-eval-harness)
# Rene: Can refer to the last shared paper on how they managed the split

In [4]:
## Obsolete: Not the best way to cluster languages, better to rely on pre-clustered data
# from sklearn.preprocessing import StandardScaler
# from sklearn.cluster import KMeans

# docs_vals = fw2_lang_dist_df[["documents"]].astype(float)

# scaler = StandardScaler()
# docs_scaled = scaler.fit_transform(docs_vals)

# n_val = 100

# kmeans = KMeans(
#     n_clusters=n_val,
#     init="k-means++",
#     n_init='auto',
#     random_state=42
# )
# fw2_lang_dist_df["lang_group"] = kmeans.fit_predict(docs_scaled)

# centroid_vals = scaler.inverse_transform(kmeans.cluster_centers_).flatten()
# label_names = [f"Group_{i}" for i in range(n_val)]          # generic names
# sorted_idx = centroid_vals.argsort()
# label_map = {sorted_idx[i]: label_names[i] for i in range(n_val)}

# fw2_lang_dist_df["lang_group"] = fw2_lang_dist_df["lang_group"].map(label_map)

# print("\nLanguages have been clustered into groups based on 'documents':")
# print(fw2_lang_dist_df[["name", "documents", "lang_group"]].head())

In [5]:
import pandas as pd

csv_path = os.path.join(PROJECT_ROOT_DIR, "data_dir/fineweb2_metadata/lang_resource_dataset.tsv")
lang_resource_df = pd.read_csv(csv_path, sep="\t")

columns_of_interest = ["lang_code", "resource_category"]

for col in columns_of_interest:
    if col in lang_resource_df.columns:
        uniques = lang_resource_df[col].dropna().unique()   # drop NaN, then get uniques
        print(f"\nUnique values in '{col}' ({len(uniques)} total):")
        print(uniques)
    else:
        print(f"\nColumn '{col}' not found in the CSV.")


Unique values in 'lang_code' (1572 total):
['bul_cyrl' 'zho_hans' 'ces_latn' ... 'tmw_latn' 'toi_latn' 'hsn_hans']

Unique values in 'resource_category' (6 total):
['high' 'medhigh' 'medlow*' 'medlow' 'low' 'not_enough']


In [6]:
# Match all the rows from fw2_lang_dist_df where "subset" has same value as "lang_code" of lang_resource_df when compared in lowercase
# For each matched row from fw2_lang_dist_df, map the "subset" value against "resource_category" from lang_resource_df

# Normalize the keys to lower‑case for a case‑insensitive join
fw2_lang_dist_df['_subset_lc'] = fw2_lang_dist_df['subset'].astype(str).str.lower()
lang_resource_df['_lang_code_lc'] = lang_resource_df['lang_code'].astype(str).str.lower()

# Perform a left join to keep all rows from the fineweb2 dataframe
fw2_lang_dist_df = fw2_lang_dist_df.merge(
    lang_resource_df[['_lang_code_lc', 'resource_category']],
    left_on='_subset_lc',
    right_on='_lang_code_lc',
    how='left'
)

# Drop rows where the mapped resource_category ends with "*"
fw2_lang_dist_df = fw2_lang_dist_df[
    ~fw2_lang_dist_df['resource_category'].str.endswith('*', na=False)
]

# Clean‑up helper columns
fw2_lang_dist_df.drop(columns=['_subset_lc', '_lang_code_lc'], inplace=True)

# Preview the result
print("\nSample rows with mapped resource categories:")
print(fw2_lang_dist_df[['subset', 'resource_category']].head())


Sample rows with mapped resource categories:
     subset resource_category
0  aai_Latn        not_enough
1  aak_Latn        not_enough
2  aau_Latn        not_enough
3  aaz_Latn        not_enough
4  aba_Latn               NaN


In [7]:
# Sort fw2_lang_dist_df by "documents" descending & find last row index for each unique value "resource_category"

fw2_lang_dist_df = fw2_lang_dist_df.sort_values(
    by="documents",
    ascending=False
).reset_index(drop=True)

print("\nDataframe sorted by 'documents' (desc):")
print(fw2_lang_dist_df[["subset", "documents", "resource_category"]].head())

# Group by `resource_category` and grab the index of the last entry in each group
last_idx_series = fw2_lang_dist_df.groupby("resource_category").apply(
    lambda grp: grp.index[-1]  # last row of the sorted group
)

# Convert to a plain dict for easy lookup
last_idx_by_category = last_idx_series.to_dict()

res_cat_tuples = []
print("\nLast row index for each `resource_category`:")
for cat, idx in last_idx_by_category.items():
    print(f"{cat!r}: {idx}")
    res_cat_tuples.append((cat, idx))


Dataframe sorted by 'documents' (desc):
     subset  documents resource_category
0  rus_Cyrl  699083579              high
1  cmn_Hani  636058984               NaN
2  deu_Latn  495964485              high
3  spa_Latn  441287261              high
4  jpn_Jpan  400138563              high

Last row index for each `resource_category`:
'high': 32
'low': 1548
'medhigh': 59
'medlow': 688
'not_enough': 1908


/tmp/ipykernel_535505/1068135741.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  last_idx_series = fw2_lang_dist_df.groupby("resource_category").apply(


In [8]:
res_cat_tuples.sort(key=lambda tup: tup[1])
print(res_cat_tuples)

[('high', 32), ('medhigh', 59), ('medlow', 688), ('low', 1548), ('not_enough', 1908)]


In [9]:
# For each item in res_cat_tuples set the "resource_category" in fw2_lang_dist_df for all rows until item[1] to the value of item[0], set item[1] as start point for the next iteration
prev_end = 0  # start of the current slice (inclusive)
for cat, end_idx in res_cat_tuples:
    # Assign the category to all rows from `prev_end` up to `end_idx` (inclusive)
    fw2_lang_dist_df.loc[prev_end:end_idx, "resource_category"] = cat
    # Next slice starts after the current end index
    prev_end = end_idx + 1

# If there are rows after the last recorded index, forward‑fill them
if prev_end < len(fw2_lang_dist_df):
    fw2_lang_dist_df.loc[prev_end:, "resource_category"] = fw2_lang_dist_df["resource_category"].ffill()

# Quick sanity check
print("\nAfter propagating `resource_category`:")
print(fw2_lang_dist_df[["subset", "resource_category"]].head(20))


After propagating `resource_category`:
      subset resource_category
0   rus_Cyrl              high
1   cmn_Hani              high
2   deu_Latn              high
3   spa_Latn              high
4   jpn_Jpan              high
5   fra_Latn              high
6   ita_Latn              high
7   por_Latn              high
8   pol_Latn              high
9   nld_Latn              high
10  ind_Latn              high
11  tur_Latn              high
12  ces_Latn              high
13  arb_Arab              high
14  vie_Latn              high
15  kor_Hang              high
16  swe_Latn              high
17  fas_Arab              high
18  ron_Latn              high
19  ukr_Cyrl              high
